# PySpark practice 6: complex types and JSON

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-06-complex-types-and-json.ipynb)

14 exercises on arrays, maps, structs, higher-order functions, exploding, nesting and flattening, JSON and VARIANT. Each exercise names the [reference entries](pyspark-reference-06-complex-types-and-json.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 6](pyspark-reference-06-complex-types-and-json.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

contacts = spark.createDataFrame([(1, "555-01", None), (2, None, "555-02"), (3, None, None)], "id int, mobile string, home string")
baskets = spark.createDataFrame(
    [(1, ["milk", "apple", "milk"], [1, 3, 2]), (2, ["bread"], [5]), (3, [], [])],
    "order_id int, items array<string>, qty array<int>",
)
payloads = spark.createDataFrame(
    [(1, '{"device": "ios", "geo": {"country": "DE"}, "tags": ["new", "promo"]}'),
     (2, '{"device": "web", "geo": {"country": "US"}}'),
     (3, "not json")],
    "event_id int, body string",
)

## Exercises

5 easy, 7 medium, 2 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Inspect and clean tag lists](#exercise-1-inspect-and-clean-tag-lists) | Easy | [`array_distinct`, `sort_array`](pyspark-reference-06-complex-types-and-json.ipynb#array_distinct-sort_array), [`size`, `array_contains`](pyspark-reference-06-complex-types-and-json.ipynb#size-array_contains) |
| 2 | [Explode order lines without losing empty orders](#exercise-2-explode-order-lines-without-losing-empty-orders) | Easy | [`explode`, `explode_outer`](pyspark-reference-06-complex-types-and-json.ipynb#explode-explode_outer) |
| 3 | [Pair parallel arrays safely](#exercise-3-pair-parallel-arrays-safely) | Medium | [`arrays_zip`](pyspark-reference-06-complex-types-and-json.ipynb#arrays_zip), [`inline`, `inline_outer`](pyspark-reference-06-complex-types-and-json.ipynb#inline-inline_outer) |
| 4 | [Totals with higher-order functions](#exercise-4-totals-with-higher-order-functions) | Medium | [`aggregate`](pyspark-reference-06-complex-types-and-json.ipynb#aggregate), [`array_max`, `array_min`, `array_sort`, `zip_with`](pyspark-reference-06-complex-types-and-json.ipynb#array_max-array_min-array_sort-zip_with) |
| 5 | [Compare permission sets](#exercise-5-compare-permission-sets) | Medium | [`array_union`, `array_intersect`, `array_except`, `arrays_overlap`](pyspark-reference-06-complex-types-and-json.ipynb#array_union-array_intersect-array_except-arrays_overlap) |
| 6 | [Look up labels in a map](#exercise-6-look-up-labels-in-a-map) | Easy | [`create_map`](pyspark-reference-06-complex-types-and-json.ipynb#create_map), [`map_keys`, `map_values`, `map_contains_key`](pyspark-reference-06-complex-types-and-json.ipynb#map_keys-map_values-map_contains_key) |
| 7 | [Clean an attribute map](#exercise-7-clean-an-attribute-map) | Medium | [`transform_keys`, `transform_values`, `map_filter`](pyspark-reference-06-complex-types-and-json.ipynb#transform_keys-transform_values-map_filter) |
| 8 | [Parse JSON payloads and count bad records](#exercise-8-parse-json-payloads-and-count-bad-records) | Medium | [`from_json`](pyspark-reference-06-complex-types-and-json.ipynb#from_json) |
| 9 | [Extract fields without a schema](#exercise-9-extract-fields-without-a-schema) | Easy | [`get_json_object`](pyspark-reference-06-complex-types-and-json.ipynb#get_json_object) |
| 10 | [Build one JSON document per order](#exercise-10-build-one-json-document-per-order) | Medium | [Building nested output with `collect_list` of structs](pyspark-reference-06-complex-types-and-json.ipynb#building-nested-output-with-collect_list-of-structs), [`to_json`](pyspark-reference-06-complex-types-and-json.ipynb#to_json) |
| 11 | [Flatten a nested source](#exercise-11-flatten-a-nested-source) | Hard | [Flattening a nested schema](pyspark-reference-06-complex-types-and-json.ipynb#flattening-a-nested-schema) |
| 12 | [Query semi-structured events with VARIANT](#exercise-12-query-semi-structured-events-with-variant) | Hard | [VARIANT: `parse_json`, `variant_get`, `try_variant_get`, `schema_of_variant`](pyspark-reference-06-complex-types-and-json.ipynb#variant-parse_json-variant_get-try_variant_get-schema_of_variant) |
| 13 | [Top two items per basket](#exercise-13-top-two-items-per-basket) | Medium | [`slice`, `array_position`, `array_remove`, `array_compact`](pyspark-reference-06-complex-types-and-json.ipynb#slice-array_position-array_remove-array_compact) |
| 14 | [Explode a map into rows](#exercise-14-explode-a-map-into-rows) | Easy | [`explode`, `explode_outer`](pyspark-reference-06-complex-types-and-json.ipynb#explode-explode_outer) |

## Exercise 1: Inspect and clean tag lists

**Difficulty:** Easy · about 5 min · **Practises:** [`array_distinct`, `sort_array`](pyspark-reference-06-complex-types-and-json.ipynb#array_distinct-sort_array), [`size`, `array_contains`](pyspark-reference-06-complex-types-and-json.ipynb#size-array_contains)

*Tags arrive with duplicates, inconsistent case and empty strings.*

For each post return `post_id`, `clean_tags` (lower-cased, trimmed, empty strings removed, distinct, sorted, joined with `,`) and `has_sale` (whether `sale` is among the clean tags). Order by `post_id`.

In [ ]:
posts = spark.createDataFrame([(1, ["Sale", " new ", "sale", ""]), (2, ["Promo"]), (3, [])], "post_id int, tags array<string>")
posts.show(truncate=False)

**Expected output**

| post_id | clean_tags | has_sale |
| --- | --- | --- |
| 1 | new,sale | true |
| 2 | promo | false |
| 3 |  | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
clean = F.sort_array(F.array_distinct(F.filter(F.transform("tags", lambda t: F.lower(F.trim(t))), lambda t: t != "")))
result = posts.select("post_id", F.array_join(clean, ",").alias("clean_tags"), F.array_contains(clean, "sale").alias("has_sale")).orderBy("post_id")
result.show()

## Exercise 2: Explode order lines without losing empty orders

**Difficulty:** Easy · about 5 min · **Practises:** [`explode`, `explode_outer`](pyspark-reference-06-complex-types-and-json.ipynb#explode-explode_outer)

*Exploding drops parents with empty lists; reports usually need them.*

From `baskets`, return one row per item as `order_id`, `item`, with a single null-item row for empty baskets. Order by `order_id`, `item` with nulls first.

In [ ]:
baskets.show()

**Expected output**

| order_id | item |
| --- | --- |
| 1 | apple |
| 1 | milk |
| 1 | milk |
| 2 | bread |
| 3 | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = baskets.select("order_id", F.explode_outer("items").alias("item")).orderBy("order_id", F.col("item").asc_nulls_first())
result.show()

## Exercise 3: Pair parallel arrays safely

**Difficulty:** Medium · about 10 min · **Practises:** [`arrays_zip`](pyspark-reference-06-complex-types-and-json.ipynb#arrays_zip), [`inline`, `inline_outer`](pyspark-reference-06-complex-types-and-json.ipynb#inline-inline_outer)

*Exploding two arrays separately creates a cross product; zip them first.*

Turn `baskets` into one row per line: `order_id`, `position` (1-based), `item`, `qty`. Exclude empty baskets. Order by `order_id`, `position`.

In [ ]:
baskets.show()

**Expected output**

| order_id | position | item | qty |
| --- | --- | --- | --- |
| 1 | 1 | milk | 1 |
| 1 | 2 | apple | 3 |
| 1 | 3 | milk | 2 |
| 2 | 1 | bread | 5 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    baskets.select("order_id", F.posexplode(F.arrays_zip("items", "qty")).alias("pos", "line"))
    .select("order_id", (F.col("pos") + 1).alias("position"), F.col("line.items").alias("item"), F.col("line.qty").alias("qty"))
    .orderBy("order_id", "position")
)
result.show()

## Exercise 4: Totals with higher-order functions

**Difficulty:** Medium · about 10 min · **Practises:** [`aggregate`](pyspark-reference-06-complex-types-and-json.ipynb#aggregate), [`array_max`, `array_min`, `array_sort`, `zip_with`](pyspark-reference-06-complex-types-and-json.ipynb#array_max-array_min-array_sort-zip_with)

*Computing inside arrays avoids an explode, a groupBy and a join back.*

For each order in `carts`, return `order_id`, `units` (sum of qty), `total` (sum of price × qty, rounded to 2) and `max_price`. Order by `order_id`.

In [ ]:
carts = spark.createDataFrame([(1, [2.5, 10.0, 4.0], [2, 1, 3]), (2, [99.99], [1])], "order_id int, price array<double>, qty array<int>")
carts.show(truncate=False)

**Expected output**

| order_id | units | total | max_price |
| --- | --- | --- | --- |
| 1 | 6 | 27.0 | 10.0 |
| 2 | 1 | 99.99 | 99.99 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = carts.select(
    "order_id",
    F.aggregate("qty", F.lit(0), lambda acc, q: acc + q).alias("units"),
    F.round(F.aggregate(F.zip_with("price", "qty", lambda p, q: p * q), F.lit(0.0), lambda acc, x: acc + x), 2).alias("total"),
    F.array_max("price").alias("max_price"),
).orderBy("order_id")
result.show()

## Exercise 5: Compare permission sets

**Difficulty:** Medium · about 10 min · **Practises:** [`array_union`, `array_intersect`, `array_except`, `arrays_overlap`](pyspark-reference-06-complex-types-and-json.ipynb#array_union-array_intersect-array_except-arrays_overlap)

*Access reviews compare granted permissions with a role's baseline.*

For each user return `user`, `missing` (baseline permissions the user lacks), `extra` (permissions beyond the baseline), each sorted and joined with `,`. Order by `user`.

In [ ]:
access = spark.createDataFrame([("ana", ["read", "write"]), ("ben", ["read", "admin", "export"]), ("cai", [])], "user string, granted array<string>")
baseline = F.array(F.lit("read"), F.lit("write"))
access.show(truncate=False)

**Expected output**

| user | missing | extra |
| --- | --- | --- |
| ana |  |  |
| ben | write | admin,export |
| cai | read,write |  |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = access.select(
    "user",
    F.array_join(F.sort_array(F.array_except(baseline, "granted")), ",").alias("missing"),
    F.array_join(F.sort_array(F.array_except("granted", baseline)), ",").alias("extra"),
).orderBy("user")
result.show()

## Exercise 6: Look up labels in a map

**Difficulty:** Easy · about 5 min · **Practises:** [`create_map`](pyspark-reference-06-complex-types-and-json.ipynb#create_map), [`map_keys`, `map_values`, `map_contains_key`](pyspark-reference-06-complex-types-and-json.ipynb#map_keys-map_values-map_contains_key)

*Small code-to-label vocabularies are often inlined as maps.*

Map each order status to a label with the map in the input cell; unmapped statuses become `Other`. Return `status` and `label` for distinct statuses, ordered by `status`.

In [ ]:
labels = F.create_map(F.lit("paid"), F.lit("Paid"), F.lit("shipped"), F.lit("On its way"), F.lit("returned"), F.lit("Refunded"))
orders.select("status").distinct().show()

**Expected output**

| status | label |
| --- | --- |
| paid | Paid |
| placed | Other |
| returned | Refunded |
| shipped | On its way |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.select("status").distinct().select("status", F.coalesce(F.element_at(labels, F.col("status")), F.lit("Other")).alias("label")).orderBy("status")
result.show()

## Exercise 7: Clean an attribute map

**Difficulty:** Medium · about 10 min · **Practises:** [`transform_keys`, `transform_values`, `map_filter`](pyspark-reference-06-complex-types-and-json.ipynb#transform_keys-transform_values-map_filter)

*Product attributes from marketplaces have messy keys and empty values.*

Lower-case the keys, drop entries with empty or null values, then return `sku` and `attrs` as JSON text (keys sorted). Order by `sku`.

In [ ]:
products_raw = spark.createDataFrame([("SKU-1", {"Color": "red", "Size": "", "FIT": "slim"}), ("SKU-2", {"Weight": None, "Material": "oak"})], "sku string, attrs map<string,string>")
products_raw.show(truncate=False)

**Expected output**

| sku | attrs |
| --- | --- |
| SKU-1 | {"color":"red","fit":"slim"} |
| SKU-2 | {"material":"oak"} |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
cleaned = F.map_filter(F.transform_keys("attrs", lambda k, v: F.lower(k)), lambda k, v: v.isNotNull() & (v != ""))
sorted_map = F.map_from_entries(F.array_sort(F.map_entries(cleaned)))
result = products_raw.select("sku", F.to_json(sorted_map).alias("attrs")).orderBy("sku")
result.show(truncate=False)

## Exercise 8: Parse JSON payloads and count bad records

**Difficulty:** Medium · about 10 min · **Practises:** [`from_json`](pyspark-reference-06-complex-types-and-json.ipynb#from_json)

*Malformed JSON becomes nulls silently; quality gates must count it.*

Parse `payloads` with schema `device string, geo struct<country string>, tags array<string>`. Return `event_id`, `device`, `country`, `tag_count` (0 when tags are missing) and `is_bad` (true when the body did not parse to any field). Order by `event_id`.

In [ ]:
payloads.show(truncate=False)

**Expected output**

| event_id | device | country | tag_count | is_bad |
| --- | --- | --- | --- | --- |
| 1 | ios | DE | 2 | false |
| 2 | web | US | 0 | false |
| 3 | null | null | 0 | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
parsed = payloads.select("event_id", F.from_json("body", "device string, geo struct<country string>, tags array<string>").alias("d"))
result = parsed.select(
    "event_id", F.col("d.device").alias("device"), F.col("d.geo.country").alias("country"),
    F.coalesce(F.size("d.tags"), F.lit(0)).alias("tag_count"),
    (F.col("d.device").isNull() & F.col("d.geo").isNull() & F.col("d.tags").isNull()).alias("is_bad"),
).orderBy("event_id")
result.show()

## Exercise 9: Extract fields without a schema

**Difficulty:** Easy · about 5 min · **Practises:** [`get_json_object`](pyspark-reference-06-complex-types-and-json.ipynb#get_json_object)

*Quick investigations pull one or two fields from raw JSON strings.*

From `logs`, return `id`, `level` and `first_sku` (the `sku` of the first item), using JSONPath. Order by `id`.

In [ ]:
logs = spark.createDataFrame([(1, '{"level": "WARN", "items": [{"sku": "SKU-1"}, {"sku": "SKU-2"}]}'), (2, '{"level": "INFO", "items": []}')], "id int, body string")
logs.show(truncate=False)

**Expected output**

| id | level | first_sku |
| --- | --- | --- |
| 1 | WARN | SKU-1 |
| 2 | INFO | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = logs.select("id", F.get_json_object("body", "$.level").alias("level"), F.get_json_object("body", "$.items[0].sku").alias("first_sku")).orderBy("id")
result.show()

## Exercise 10: Build one JSON document per order

**Difficulty:** Medium · about 15 min · **Practises:** [Building nested output with `collect_list` of structs](pyspark-reference-06-complex-types-and-json.ipynb#building-nested-output-with-collect_list-of-structs), [`to_json`](pyspark-reference-06-complex-types-and-json.ipynb#to_json)

*APIs and document stores want nested records, not flat rows.*

For orders 1 and 2, return `order_id` and `doc`: JSON with `order_id`, `status` and `lines` (array of `{sku, qty}` sorted by `sku`). Order by `order_id`.

In [ ]:
order_items.where("order_id <= 2").show()

**Expected output**

| order_id | doc |
| --- | --- |
| 1 | {"order_id":1,"status":"paid","lines":[{"sku":"SKU-4001","qty":2},{"sku":"SKU-4007","qty":1}]} |
| 2 | {"order_id":2,"status":"shipped","lines":[{"sku":"SKU-4002","qty":1},{"sku":"SKU-4009","qty":2}]} |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
lines = order_items.groupBy("order_id").agg(F.array_sort(F.collect_list(F.struct("sku", "qty"))).alias("lines"))
result = (
    orders.where("order_id <= 2").join(lines, "order_id")
    .select("order_id", F.to_json(F.struct("order_id", "status", "lines")).alias("doc"))
    .orderBy("order_id")
)
result.show(truncate=False)

## Exercise 11: Flatten a nested source

**Difficulty:** Hard · about 15 min · **Practises:** [Flattening a nested schema](pyspark-reference-06-complex-types-and-json.ipynb#flattening-a-nested-schema)

*Warehouses and BI tools need flat columns.*

Flatten every struct field of `deep` into `parent_child` columns (arrays stay as they are). Return the column names in order as `position` and `column`.

In [ ]:
deep = spark.createDataFrame([], "order_id int, meta struct<customer:struct<id:int,name:string>,channel:string>, tags array<string>, ship struct<city:string>")
deep.printSchema()

**Expected output**

| position | column |
| --- | --- |
| 1 | order_id |
| 2 | meta_customer_id |
| 3 | meta_customer_name |
| 4 | meta_channel |
| 5 | tags |
| 6 | ship_city |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
from pyspark.sql.types import StructType

def flat_columns(schema, prefix=""):
    out = []
    for field in schema.fields:
        path = f"{prefix}{field.name}"
        if isinstance(field.dataType, StructType):
            out += flat_columns(field.dataType, path + ".")
        else:
            out.append(path)
    return out

flat = deep.select(*[F.col(p).alias(p.replace(".", "_")) for p in flat_columns(deep.schema)])
result = spark.createDataFrame(list(enumerate(flat.columns, 1)), "position int, column string")
result.show()

## Exercise 12: Query semi-structured events with VARIANT

**Difficulty:** Hard · about 15 min · **Practises:** [VARIANT: `parse_json`, `variant_get`, `try_variant_get`, `schema_of_variant`](pyspark-reference-06-complex-types-and-json.ipynb#variant-parse_json-variant_get-try_variant_get-schema_of_variant)

*Event payloads differ by event type; VARIANT avoids one schema per type.*

Parse `raw` with `parse_json` and return `id`, `type`, `amount` (decimal(10,2), null when absent or not numeric) and `page`. Order by `id`.

In [ ]:
raw = spark.createDataFrame([(1, '{"type": "click", "meta": {"page": "/home"}}'), (2, '{"type": "buy", "amount": "12.50"}'), (3, '{"type": "buy", "amount": "n/a"}')], "id int, body string")
raw.show(truncate=False)

**Expected output**

| id | type | amount | page |
| --- | --- | --- | --- |
| 1 | click | null | /home |
| 2 | buy | 12.50 | null |
| 3 | buy | null | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
v = raw.select("id", F.parse_json("body").alias("v"))
result = v.select(
    "id", F.variant_get("v", "$.type", "string").alias("type"),
    F.try_variant_get("v", "$.amount", "decimal(10,2)").alias("amount"),
    F.try_variant_get("v", "$.meta.page", "string").alias("page"),
).orderBy("id")
result.show()

## Exercise 13: Top two items per basket

**Difficulty:** Medium · about 10 min · **Practises:** [`slice`, `array_position`, `array_remove`, `array_compact`](pyspark-reference-06-complex-types-and-json.ipynb#slice-array_position-array_remove-array_compact)

*Per-row top-N without a window: sort the array and slice it.*

Each basket has `items` and parallel `qty`. Return `order_id` and `top2`: the two items with the highest quantity (ties broken by item name), joined with `,`. Order by `order_id`.

In [ ]:
carts2 = spark.createDataFrame([(1, ["milk", "apple", "bread"], [1, 3, 3]), (2, ["tea"], [2])], "order_id int, items array<string>, qty array<int>")
carts2.show(truncate=False)

**Expected output**

| order_id | top2 |
| --- | --- |
| 1 | apple,bread |
| 2 | tea |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
pairs = F.transform(F.arrays_zip("qty", "items"), lambda p: F.struct((-p["qty"]).alias("neg_qty"), p["items"].alias("item")))
top2 = F.transform(F.slice(F.array_sort(pairs), 1, 2), lambda p: p["item"])
result = carts2.select("order_id", F.array_join(top2, ",").alias("top2")).orderBy("order_id")
result.show()

## Exercise 14: Explode a map into rows

**Difficulty:** Easy · about 5 min · **Practises:** [`explode`, `explode_outer`](pyspark-reference-06-complex-types-and-json.ipynb#explode-explode_outer)

*Attribute maps are unpivoted into key/value rows for EAV tables.*

Explode each product's `attrs` map into rows `sku`, `key`, `value`, ordered by `sku`, `key`.

In [ ]:
attrs = spark.createDataFrame([("SKU-1", {"color": "red", "size": "M"}), ("SKU-2", {"material": "oak"})], "sku string, attrs map<string,string>")
attrs.show(truncate=False)

**Expected output**

| sku | key | value |
| --- | --- | --- |
| SKU-1 | color | red |
| SKU-1 | size | M |
| SKU-2 | material | oak |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = attrs.select("sku", F.explode("attrs").alias("key", "value")).orderBy("sku", "key")
result.show()

**Next:** look anything up in [PySpark reference 6: complex types and JSON](pyspark-reference-06-complex-types-and-json.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).